# 전체어휘 LDA

- 대응 코드: `lda_full.py`
- 언어별 전체어휘 토큰과 토픽 수 후보를 확인.
- Okt/Janome 및 모델 학습이 필요하므로 전체 재실행은 선택.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 작업 폴더가 저장소 루트이거나 노트북 폴더여도 analysis/를 찾는다.
candidates = [Path.cwd(), *Path.cwd().parents]
analysis_root = next((candidate for base in candidates
                      for candidate in (base, base / "code" / "analysis", base / "ESG_comp" / "code" / "analysis")
                      if (candidate / "common" / "config.py").is_file()), None)
if analysis_root is None:
    raise FileNotFoundError("ESG_comp/code/analysis 폴더에서 노트북을 열어 주세요.")
if not os.environ.get("ESG_RESEARCH_ROOT"):
    raise RuntimeError("비공개 입력 위치를 ESG_RESEARCH_ROOT 환경변수로 먼저 지정하세요.")
sys.path.insert(0, str(analysis_root))
from common import config as cfg
from common.io_common import load_enriched


def show_csv(path, rows=10, columns=None):
    path = Path(path)
    if not path.is_file():
        print(f"저장 결과 없음: {path}\n입력 자료와 앞 단계를 확인한 뒤 실행하세요.")
        return None
    frame = pd.read_csv(path)
    display(frame[columns].head(rows) if columns else frame.head(rows))
    print(f"{path.name}: {len(frame):,}행")
    return frame


In [ ]:
LANG = "kr"
RUN_TOKENIZE = False
RUN_REPORT = False
PREVIEW_ROWS = 10

In [ ]:
from stage4_lda import lda_full as step
if LANG not in cfg.LANGS:
    raise ValueError(f"LANG은 {cfg.LANGS} 중 하나여야 합니다.")
print("기준 파라미터:", step.PARAMS)
show_csv(cfg.OUTPUT_DIR / f"docs_{LANG}.csv", PREVIEW_ROWS)
print("토큰화는 RUN_TOKENIZE=True로 선택하세요. 한국어는 Okt, 일본어는 Janome이 필요합니다.")

In [ ]:
if RUN_TOKENIZE:
    doc_tokens = step.build_doc_tokens(LANG)
    print(f"토큰화 문서 {len(doc_tokens):,}건")
    display(doc_tokens.head(PREVIEW_ROWS))
if RUN_REPORT:
    step.build_stage4_full_report(LANG)
if not (RUN_TOKENIZE or RUN_REPORT):
    print("전체어휘 LDA 실행은 RUN_TOKENIZE/RUN_REPORT에서 선택하세요.")

In [ ]:
out_dir = cfg.OUTPUT_DIR / LANG / "단계4_전체어휘_LDA"
show_csv(out_dir / f"lda_full_summary_{LANG}.csv", PREVIEW_ROWS)
show_csv(out_dir / f"topic_terms_{LANG}.csv", PREVIEW_ROWS)
show_csv(out_dir / f"topic_samples_{LANG}.csv", PREVIEW_ROWS)